# Code Generator

A simple code generator using Gradio and DeepSeek API (via OpenAI-compatible client).

In [56]:
from dotenv import load_dotenv
import os
import json

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
MODEL = os.getenv("MODEL", "deepseek-v4-flash")

print(f"Base URL: {DEEPSEEK_BASE_URL}")
print(f"API Key set: {bool(DEEPSEEK_API_KEY)}")

Base URL: https://api.deepseek.com/v1
API Key set: True


In [57]:
from openai import OpenAI

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

print("OpenAI client initialized (pointed at DeepSeek API)")

OpenAI client initialized (pointed at DeepSeek API)


In [58]:
system_prompt = """
You are a helpful assistant that can create a web application based on the user request. You can use any technology stack you want, but you should try to use the most appropriate one for the user's request. You should also try to keep the code as simple as possible, while still being functional. If the user asks for a specific technology stack, you should use that stack. You should also try to include comments in your code to explain what each part of the code does.

The code that you generate will be written into a certain folders so don't put the generated code into your answer for the user.
"""

In [59]:
write_file_tool = {
    "name": "write_file",
    "description": "Write a file to the system.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {"type": "string", "description": "The name of the file to write, example 'index.html'."},
            "filepath": {"type": "string", "description": "The relative file path to write to, example 'src/index.html'. This path will be joined with the determined base directory for the project."},
            "content": {"type": "string", "description": "The content to write to the file."},
        },
        "required": ["filename", "filepath", "content"],
    },
}

tools = [
    {"type": "function", "function": write_file_tool},
]

In [60]:
def write_file(projectId, filename, filepath, content):
    full_path = os.path.join("projects", projectId, filepath)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    
    with open(full_path, "w") as f:
        f.write(content)

    return f"File {filename} written to {full_path}"

In [61]:
def create_project_id():
    import random
    import string

    return ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))

In [62]:
def handle_tool_calls(tool_calls, projectId):
    responses = []

    for call in tool_calls:
        if call.function.name == "write_file":
            arguments = json.loads(call.function.arguments)
            filename = arguments.get("filename")
            filepath = arguments.get("filepath")
            content = arguments.get("content")
            result = write_file(projectId, filename, filepath, content)

            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": call.id
            })

    return responses

In [63]:
import gradio as gr

def createChatSession(projectId):
    def predict(message, history):
        messages = [
            {"role": "system", "content": system_prompt}, 
        ]

        for item in history:
            messages.append({"role": item["role"], "content": item["content"]})

        messages.append({"role": "user", "content": message})

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        while response.choices[0].finish_reason=="tool_calls":
            tool_message = response.choices[0].message
            tool_responses = handle_tool_calls(tool_message.tool_calls, projectId)

            messages.append(tool_message)
            messages.extend(tool_responses)

            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
            )

        assistant_reply = response.choices[0].message.content

        print(f"Reply assistant_reply: {assistant_reply}")
        
        return assistant_reply
    
    return predict 

In [64]:
projectId = create_project_id()
demo = gr.ChatInterface(
    fn=createChatSession(projectId),
    title="AI Code Generator",
    description="Generate code using DeepSeek AI",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


/home/pramindanata/Code/ai-code-generator/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Reply assistant_reply: Your portfolio is ready! Here's a summary of what I created:

### 📁 Project Structure

```
portfolio/
├── index.html          # Main HTML page
├── src/
│   ├── style.css       # All styles (responsive, modern)
│   └── script.js       # Interactivity & dynamic content
```

### ✨ Features

| Section | Details |
|---|---|
| **Hero** | Full-screen gradient intro with name, tagline, and CTA buttons |
| **About** | Short bio with an avatar placeholder |
| **Skills** | 8 tech skills rendered dynamically from JavaScript (HTML, CSS, JS, React, Node, Python, SQL, Git) |
| **Projects** | 3 project cards with icons, descriptions, and tech tags |
| **Contact** | Form with validation + toast notification on submit |
| **Navigation** | Fixed header with smooth-scroll links + **mobile hamburger menu** |
| **Animations** | Scroll-reveal fade-in effect for cards |

### 🎨 Design Highlights
- **Purple accent** (`#6c63ff`) theme throughout
- **Responsive** — works on mobile, tablet, 

/home/pramindanata/Code/ai-code-generator/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
